In [1]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
import re
import csv
from pathlib import Path

In [5]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [6]:
from scipy.sparse import load_npz
from rank_bm25 import BM25Okapi

In [7]:
DATASET_FOLDER = Path(
    "/Users/megan/Downloads/PMC_Patients")

In [8]:
OUTPUT_FOLDER = DATASET_FOLDER / "tfidf_output"

MATRIX_FILE = OUTPUT_FOLDER / "tfidf_matrix.npz"
METADATA_FILE = OUTPUT_FOLDER / "document_metadata.csv"
OUTPUT_FILE = OUTPUT_FOLDER / "task2_similar_patients.json"

In [9]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [10]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()

    processed_words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(processed_words)

In [11]:
records = []
documents = []

In [12]:
json_files = sorted(DATASET_FOLDER.rglob("*.json"))

In [10]:
for file_path in json_files:

    if OUTPUT_FOLDER in file_path.parents:
        continue

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    items = data if isinstance(data, list) else [data]

    for record in items:
        if not isinstance(record, dict):
            continue

        title = record.get("title", "")
        patient_text = record.get("patient", "")

        if not isinstance(title, str):
            title = ""

        if not isinstance(patient_text, str):
            patient_text = ""

        original_text = f"{title} {patient_text}".strip()

        if not original_text:
            continue

        processed_text = preprocess_text(original_text)

        records.append({
            "patient_uid": str(record.get("patient_uid", "")),
            "patient_id": str(record.get("patient_id", "")),
            "filename": str(
                file_path.relative_to(DATASET_FOLDER)
            )
        })

        documents.append(processed_text)

In [11]:
tfidf_matrix = load_npz(MATRIX_FILE)

In [12]:
with open(METADATA_FILE, "r", encoding="utf-8") as file:
    metadata = list(csv.DictReader(file))

In [13]:
if len(records) != tfidf_matrix.shape[0]:
    raise ValueError(
        "The number of records does not match the TF-IDF "
        "matrix rows. Check the dataset order and Task 1 files."
    )

In [14]:
if len(metadata) != len(records):
    raise ValueError(
        "Metadata row count does not match the records."
    )

In [15]:
for i, record in enumerate(records):
    if record["filename"] != metadata[i]["filename"]:
        raise ValueError(
            f"Record order mismatch at row {i}. "
            "Do not use these results until the order is fixed."
        )

In [16]:
if any(not record["patient_uid"] for record in records):
    raise ValueError("Some records are missing patient_uid.")

In [17]:
patient_uids = [record["patient_uid"] for record in records]

In [18]:
if len(set(patient_uids)) != len(patient_uids):
    raise ValueError(
        "Duplicate patient_uid values found. "
        "The output dictionary requires unique keys."
    )

In [19]:
number_of_patients = len(records)

In [20]:
# we want the top five similar patient documents
if number_of_patients < 6:
    raise ValueError(
        "At least six records are needed to retrieve five "
        "other patients for every query."
    )

In [21]:
print("Records:", number_of_patients)
print("TF-IDF matrix:", tfidf_matrix.shape)

Records: 8579
TF-IDF matrix: (8579, 61302)


In [22]:
# TF-IDF retrieval using cosine similarity first

In [23]:
tfidf_results = {}

In [24]:
# don't want to process all at once due to large corpus
batch_size = 128

In [25]:
for start in range(0, number_of_patients, batch_size):
    end = min(start + batch_size, number_of_patients)


    similarity_batch = (
        tfidf_matrix[start:end] @ tfidf_matrix.T
    ).toarray()

    for local_index, scores in enumerate(similarity_batch):
        query_index = start + local_index

        # exclude the patient from their own document
        scores[query_index] = float("-inf")

        #five highest-scoring other patients
        top_indices = scores.argsort()[-5:][::-1]

        tfidf_results[query_index] = [
            patient_uids[i] for i in top_indices
        ]

    print(f"TF-IDF processed: {end}/{number_of_patients}")

TF-IDF processed: 128/8579
TF-IDF processed: 256/8579
TF-IDF processed: 384/8579
TF-IDF processed: 512/8579
TF-IDF processed: 640/8579
TF-IDF processed: 768/8579
TF-IDF processed: 896/8579
TF-IDF processed: 1024/8579
TF-IDF processed: 1152/8579
TF-IDF processed: 1280/8579
TF-IDF processed: 1408/8579
TF-IDF processed: 1536/8579
TF-IDF processed: 1664/8579
TF-IDF processed: 1792/8579
TF-IDF processed: 1920/8579
TF-IDF processed: 2048/8579
TF-IDF processed: 2176/8579
TF-IDF processed: 2304/8579
TF-IDF processed: 2432/8579
TF-IDF processed: 2560/8579
TF-IDF processed: 2688/8579
TF-IDF processed: 2816/8579
TF-IDF processed: 2944/8579
TF-IDF processed: 3072/8579
TF-IDF processed: 3200/8579
TF-IDF processed: 3328/8579
TF-IDF processed: 3456/8579
TF-IDF processed: 3584/8579
TF-IDF processed: 3712/8579
TF-IDF processed: 3840/8579
TF-IDF processed: 3968/8579
TF-IDF processed: 4096/8579
TF-IDF processed: 4224/8579
TF-IDF processed: 4352/8579
TF-IDF processed: 4480/8579
TF-IDF processed: 4608/8579

In [13]:
# I picked BM25 for the next method and wanted to print 500 at at 

In [28]:
tokenized_documents = [
    document.split() for document in documents
]

In [29]:
bm25 = BM25Okapi(tokenized_documents)

In [30]:
bm25_results = {}

In [36]:
for query_index, query_tokens in enumerate(tokenized_documents):

    scores = bm25.get_scores(query_tokens)

    # Exclude the query patient.
    scores[query_index] = float("-inf")

    top_indices = scores.argsort()[-5:][::-1]

    bm25_results[query_index] = [
        patient_uids[i] for i in top_indices
    ]

    if (query_index + 1) % 500 == 0:
        print(
            f"BM25 processed: "
            f"{query_index + 1}/{number_of_patients}"
        )

BM25 processed: 500/8579
BM25 processed: 1000/8579
BM25 processed: 1500/8579
BM25 processed: 2000/8579
BM25 processed: 2500/8579
BM25 processed: 3000/8579
BM25 processed: 3500/8579
BM25 processed: 4000/8579
BM25 processed: 4500/8579
BM25 processed: 5000/8579
BM25 processed: 5500/8579
BM25 processed: 6000/8579
BM25 processed: 6500/8579
BM25 processed: 7000/8579
BM25 processed: 7500/8579
BM25 processed: 8000/8579
BM25 processed: 8500/8579


In [37]:
results = {}

In [38]:
for i, record in enumerate(records):

    results[record["patient_uid"]] = {
        "patient_id": record["patient_id"],
        "tfidf": tfidf_results[i],
        "bm25": bm25_results[i]
    }

In [39]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(results, file, indent=2, ensure_ascii=False)

In [40]:
print("\nTask 2 completed!")
print("Patients processed:", len(results))
print("Output saved to:", OUTPUT_FILE)


Task 2 completed!
Patients processed: 8579
Output saved to: /Users/megan/Downloads/PMC_Patients/tfidf_output/task2_similar_patients.json


In [41]:
# one example to double-check it works

first_uid = patient_uids[0]
print("\nExample result:")
print(json.dumps(
    {first_uid: results[first_uid]},
    indent=2
))


Example result:
{
  "1192821-2": {
    "patient_id": "88468",
    "tfidf": [
      "8376335-1",
      "3271401-1",
      "5987608-1",
      "7658281-1",
      "6366377-1"
    ],
    "bm25": [
      "8376335-1",
      "5987608-1",
      "7658281-1",
      "3271401-1",
      "7694143-1"
    ]
  }
}
